# Week 5 — Cloud Infrastructure Orchestration

We now leave the single-node setting and target ephemeral, possibly preemptible cloud GPU clusters. The deliverable: a **Dockerized training image** that can be launched on RunPod, Lambda Labs, or Vast.ai with a single CLI command, and a checkpoint strategy that survives spot preemptions.

## Learning objectives

1. Compose a CUDA-12.4 / PyTorch 2.4 / DeepSpeed Docker image with reproducible layer ordering.
2. Configure WandB for live monitoring of a remote training job.
3. Derive and apply the **optimal checkpoint frequency** under preemption.
4. SSH-tunnel into a running pod for live debugging.


## 1. Anatomy of a Reproducible CUDA Image

A production-grade Dockerfile has four layers, ordered from least to most volatile (for maximum cache reuse):

1. **Base.** `nvidia/cuda:12.4.1-cudnn-devel-ubuntu22.04` — pinned exactly.
2. **System packages.** `git`, `build-essential`, `openssh-client`, NCCL.
3. **Python environment.** Specific PyTorch, DeepSpeed, and CUDA-matching pip wheels.
4. **Application code.** The repository itself, copied last so that code edits don't invalidate earlier cache layers.


In [ ]:
# docker/Dockerfile (excerpt)
FROM nvidia/cuda:12.4.1-cudnn-devel-ubuntu22.04 AS base

ARG PYTHON_VERSION=3.11
ENV DEBIAN_FRONTEND=noninteractive PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y --no-install-recommends \
        git build-essential ca-certificates curl openssh-client \
        python${PYTHON_VERSION} python${PYTHON_VERSION}-dev python3-pip && \
    rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir \
        torch==2.4.0 --index-url https://download.pytorch.org/whl/cu124 && \
    pip install --no-cache-dir \
        deepspeed==0.15.0 wandb rich pyyaml numpy

WORKDIR /workspace
COPY . /workspace
RUN pip install --no-cache-dir -e .

ENTRYPOINT ["bash", "/workspace/docker/entrypoint.sh"]


## 2. Launching on RunPod / Lambda / Vast.ai

The three providers expose slightly different surfaces, but the workflow is the same:

```bash
# 1. Push the image to a registry
docker push ghcr.io/HAYDARKILIC/hpc-llm-forge:latest

# 2. Submit the job
runpod create-pod \\
    --gpu-type H100 \\
    --gpu-count 8 \\
    --image ghcr.io/HAYDARKILIC/hpc-llm-forge:latest \\
    --env WANDB_API_KEY=$WANDB_API_KEY \\
    --command "make launch-deepspeed CONFIG=configs/training/gpt_1b3.yaml"
```

The `cloud_launcher.py` module wraps these calls behind a single Python API so that the capstone pipeline is provider-agnostic.


## 3. Spot-Resilient Checkpointing

Spot/preemptible instances are 60–80% cheaper but can be terminated with $\sim 30$ s notice. Let

* $\lambda$ = preemption rate (events / hour),
* $C_{\text{ckpt}}$ = checkpoint cost (seconds),
* $t$ = current time since last checkpoint.

Expected work lost per preemption is $\mathbb{E}[t] = 1 / (2 f)$ where $f$ is the checkpoint frequency. Minimising **(work lost + checkpoint cost)** over $f$ yields

$$
f^{*} \;=\; \sqrt{\,\lambda \,/\, C_{\text{ckpt}}\,}.
$$

That is, the optimal interval scales as the geometric mean of the expected preemption time and the checkpoint cost.


In [ ]:
# %% Self-tuning checkpoint manager
from hpcllmforge.orchestration.checkpoint_manager import CheckpointManager

ckpt = CheckpointManager(out_dir='./checkpoints', initial_interval_sec=600)

for step in range(10_000):
    # ... training step ...
    if ckpt.should_checkpoint():
        # save model + optimizer + RNG state ...
        pass

# When a SIGTERM is received from the preemption notice:
import signal, time
signal.signal(signal.SIGTERM, lambda *_: ckpt.record_preemption(time.time()))


## 4. Remote Observability via WandB + SSH

* **WandB.** Every metric, every config, every system stat (GPU util, HBM use, NVLink traffic) is logged to a single web dashboard.
* **SSH tunnel.** `ssh -L 8265:localhost:8265 <pod>` exposes `nvidia-smi dmon`, `nvtop`, and PyTorch's TensorBoard locally.

```python
import wandb
wandb.init(project='hpc-llm-forge', config=config, name='gpt-1b3-h100x8')
wandb.watch(model, log_freq=100)
```


## 5. Exercises

1. Build and push the image. Time the cold-start (no cache) vs warm-start (cache present) build.
2. Submit the same training job to two providers (e.g. RunPod and Lambda). Compare wallclock cost and throughput.
3. Simulate preemption by sending SIGTERM at a random time and verify that training resumes from the latest checkpoint with no metric discontinuity.
